In [ ]:
import numpy as np
import pandas as pd

# Galaxy Image Classification — Numerical Results

**Single source of truth** for all computed quantities cited in the report. Every number in the report text must trace to a cell in this notebook.

In [ ]:
# Load all results
split_data = np.load("split_indices.npz")
resnet_data = np.load("resnet_result.npz")
custom_data = np.load("custom_result.npz", allow_pickle=True)
sched_data = np.load("scheduler_result.npz")
aug_data = np.load("augmented_result.npz")
eval_data = np.load("evaluation_results.npz")
label_data = np.load("galaxy_zoo_labels.npz")

## Dataset Summary

In [ ]:
n_total = len(label_data["labels"])
n_train = len(split_data["train_idx"])
n_val = len(split_data["val_idx"])

dataset_summary = pd.DataFrame({
    "Quantity": ["Total galaxies", "Training set", "Validation set", "Number of labels"],
    "Value": [n_total, n_train, n_val, label_data["labels"].shape[1]],
})
dataset_summary

## Model Performance Summary

In [ ]:
performance = pd.DataFrame({
    "Model": [
        "Baseline (mean)",
        "Custom CNN",
        "ResNet-18",
        "ResNet-18 + LR Scheduler",
        "ResNet-18 + Aug + LR Scheduler",
    ],
    "Best Val RMSE": [
        float(split_data["baseline_val_rmse"]),
        float(custom_data["best_val_loss"]),
        float(resnet_data["best_val_loss"]),
        float(sched_data["best_val_loss"]),
        float(aug_data["best_val_loss"]),
    ],
    "Best Epoch": [
        "N/A",
        int(custom_data["best_epoch"]) + 1,
        int(resnet_data["best_epoch"]) + 1,
        int(sched_data["best_epoch"]) + 1,
        int(aug_data["best_epoch"]) + 1,
    ],
    "Parameters": [
        0,
        int(custom_data["n_parameters"]),
        int(resnet_data["n_parameters"]),
        int(sched_data["n_parameters"]),
        int(aug_data["n_parameters"]),
    ],
})
performance

## Per-Label Performance (Best Model)

In [ ]:
from ugdatalab.models.galaxy_zoo.constants import LABEL_COLUMNS, LABEL_DESCRIPTIVE

val_true = eval_data["val_true_labels"]
val_pred = eval_data["val_pred_labels"]

per_label = []
for i, col in enumerate(LABEL_COLUMNS):
    resid = val_pred[:, i] - val_true[:, i]
    per_label.append({
        "Label": LABEL_DESCRIPTIVE[col],
        "Bias": f"{np.mean(resid):+.4f}",
        "Scatter": f"{np.std(resid):.4f}",
        "RMSE": f"{np.sqrt(np.mean(resid**2)):.4f}",
    })

per_label_df = pd.DataFrame(per_label)
per_label_df

## Merger Fraction

In [ ]:
merger_fraction = float(eval_data["merger_fraction"])
n_test = len(eval_data["test_galaxy_ids"])

print(f"Test set size: {n_test}")
print(f"Estimated merger fraction (mean prob): {merger_fraction*100:.2f}%")
print()
print("Comparison to Lotz et al. 2011:")
print("  Observed major merger rate at z≈0: ~0.01-0.03 Gyr^-1")
print("  Merger observability timescale: ~0.5-1 Gyr")
print(f"  Expected merger fraction: ~0.5-3%")
print(f"  Our estimate: {merger_fraction*100:.2f}%")